# IPL Win Probability Predictor - ML Workflow

This notebook contains the complete machine learning workflow to train a logistic regression model for predicting the win probability of an IPL team during a run chase.

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 1. Dataset Loading

In [ ]:
try:
    matches = pd.read_csv('data/matches.csv')
    deliveries = pd.read_csv('data/deliveries.csv')
    print("Datasets loaded successfully!")
    # deliveries dataset may not have total_runs, we create it
    deliveries['total_runs'] = deliveries['batsman_runs'] + deliveries['extras']
except FileNotFoundError:
    print("Please ensure 'matches.csv' and 'deliveries.csv' are present in the 'data/' directory.")
    # Dummy data structures just so the code compiles if files are missing
    matches = pd.DataFrame(columns=['matchId', 'city', 'winner', 'team1', 'team2'])
    deliveries = pd.DataFrame(columns=['matchId', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman_runs', 'extras', 'total_runs', 'player_dismissed'])

## 2. Data Preprocessing & Feature Engineering

We need to create the following features for our model:
- `batting_team`
- `bowling_team`
- `runs_left`
- `balls_left`
- `wickets_left`
- `target`
- `crr` (Current Run Rate)
- `rrr` (Required Run Rate)
- `result` (1 for win, 0 for loss)

In [ ]:
if not matches.empty and not deliveries.empty:
    # Calculate total score of first innings to set the target for second innings
    total_score_df = deliveries.groupby(['matchId', 'inning']).sum(numeric_only=True)['total_runs'].reset_index()
    total_score_df = total_score_df[total_score_df['inning'] == 1]
    
    # Merge target with match info
    match_df = matches.merge(total_score_df[['matchId', 'total_runs']], on='matchId')
    
    # Standardize team names
    match_df['team1'] = match_df['team1'].str.replace('Delhi Daredevils', 'Delhi Capitals')
    match_df['team2'] = match_df['team2'].str.replace('Delhi Daredevils', 'Delhi Capitals')
    match_df['team1'] = match_df['team1'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
    match_df['team2'] = match_df['team2'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
    match_df['team1'] = match_df['team1'].str.replace('Kings XI Punjab', 'Punjab Kings')
    match_df['team2'] = match_df['team2'].str.replace('Kings XI Punjab', 'Punjab Kings')
    
    teams = [
        'Sunrisers Hyderabad',
        'Mumbai Indians',
        'Royal Challengers Bangalore',
        'Kolkata Knight Riders',
        'Punjab Kings',
        'Chennai Super Kings',
        'Rajasthan Royals',
        'Delhi Capitals'
    ]
    
    match_df = match_df[match_df['team1'].isin(teams)]
    match_df = match_df[match_df['team2'].isin(teams)]
    
    # Keep relevant columns and clean missing values
    match_df = match_df[['matchId', 'city', 'winner', 'total_runs']]
    
    # Merge with delivery data
    delivery_df = match_df.merge(deliveries, on='matchId')
    
    # Standardize team names in delivery_df as well
    delivery_df['batting_team'] = delivery_df['batting_team'].str.replace('Delhi Daredevils', 'Delhi Capitals')
    delivery_df['bowling_team'] = delivery_df['bowling_team'].str.replace('Delhi Daredevils', 'Delhi Capitals')
    delivery_df['batting_team'] = delivery_df['batting_team'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
    delivery_df['bowling_team'] = delivery_df['bowling_team'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
    delivery_df['batting_team'] = delivery_df['batting_team'].str.replace('Kings XI Punjab', 'Punjab Kings')
    delivery_df['bowling_team'] = delivery_df['bowling_team'].str.replace('Kings XI Punjab', 'Punjab Kings')
    
    # Keep only second innings data (run chase)
    delivery_df = delivery_df[delivery_df['inning'] == 2]
    
    # Calculate current score
    delivery_df['current_score'] = delivery_df.groupby('matchId')['total_runs_y'].cumsum()
    
    # Calculate runs left and balls left
    delivery_df['runs_left'] = delivery_df['total_runs_x'] - delivery_df['current_score'] + 1
    delivery_df['balls_left'] = 120 - (delivery_df['over'] * 6 + delivery_df['ball'])
    
    # Calculate wickets left
    delivery_df['player_dismissed'] = delivery_df['player_dismissed'].fillna("0")
    delivery_df['player_dismissed'] = delivery_df['player_dismissed'].apply(lambda x: x if x == "0" else "1")
    delivery_df['player_dismissed'] = delivery_df['player_dismissed'].astype('int')
    wickets = delivery_df.groupby('matchId')['player_dismissed'].cumsum().values
    delivery_df['wickets_left'] = 10 - wickets
    
    # Current Run Rate (CRR)
    delivery_df['crr'] = (delivery_df['current_score'] * 6) / (120 - delivery_df['balls_left'])
    
    # Required Run Rate (RRR)
    delivery_df['rrr'] = (delivery_df['runs_left'] * 6) / delivery_df['balls_left']
    
    # Target
    delivery_df['target'] = delivery_df['total_runs_x'] + 1
    
    # Result (1 for win, 0 for loss)
    # Also standardizing winner column
    delivery_df['winner'] = delivery_df['winner'].str.replace('Delhi Daredevils', 'Delhi Capitals')
    delivery_df['winner'] = delivery_df['winner'].str.replace('Deccan Chargers', 'Sunrisers Hyderabad')
    delivery_df['winner'] = delivery_df['winner'].str.replace('Kings XI Punjab', 'Punjab Kings')
    
    def result(row):
        return 1 if row['batting_team'] == row['winner'] else 0
    
    delivery_df['result'] = delivery_df.apply(result, axis=1)
    
    # Final Dataset
    final_df = delivery_df[['batting_team', 'bowling_team', 'runs_left', 'balls_left', 'wickets_left', 'target', 'crr', 'rrr', 'result']]
    
    # Handle edge cases (like balls_left = 0 leading to infinite rrr)
    final_df = final_df[final_df['balls_left'] != 0]
    final_df = final_df.dropna()
    
    print("Data Preprocessing Complete! Shape of final dataset:", final_df.shape)
else:
    print("Skipping preprocessing as datasets were not found.")
    final_df = pd.DataFrame(columns=['batting_team', 'bowling_team', 'runs_left', 'balls_left', 'wickets_left', 'target', 'crr', 'rrr', 'result'])

## 3. Model Training

We will use a Pipeline with a ColumnTransformer to One-Hot Encode the categorical variables (`batting_team`, `bowling_team`) and then train a Logistic Regression model.

In [ ]:
if not final_df.empty:
    # Split data
    X = final_df.drop('result', axis=1)
    y = final_df['result']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Define categorical columns
    categorical_cols = ['batting_team', 'bowling_team']
    
    # Create ColumnTransformer
    trf = ColumnTransformer([
        ('trf', OneHotEncoder(sparse_output=False, drop='first'), categorical_cols)
    ], remainder='passthrough')
    
    # Create Pipeline
    pipe = Pipeline(steps=[
        ('step1', trf),
        ('step2', LogisticRegression(solver='liblinear'))
    ])
    
    # Train model
    pipe.fit(X_train, y_train)
    
    print("Model trained successfully!")
    
    # Evaluate model
    y_pred = pipe.predict(X_test)
    print("Accuracy Score:", accuracy_score(y_test, y_pred))
else:
    print("Skipping model training as dataset is empty.")
    pipe = None

## 4. Model Evaluation & Sample Prediction

In [ ]:
if pipe is not None:
    # Example Prediction
    sample_input = pd.DataFrame({
        'batting_team': ['Mumbai Indians'],
        'bowling_team': ['Chennai Super Kings'],
        'runs_left': [45],
        'balls_left': [30],
        'wickets_left': [6],
        'target': [180],
        'crr': [9.0],
        'rrr': [9.0]
    })
    
    prob = pipe.predict_proba(sample_input)[0]
    print(f"Loss Probability: {prob[0]*100:.2f}%")
    print(f"Win Probability: {prob[1]*100:.2f}%")

## 5. Save Model using Pickle

We save the entire pipeline so that it includes both the OneHotEncoder and the Logistic Regression model. This makes deployment easy.

In [ ]:
if pipe is not None:
    with open('pipe.pkl', 'wb') as f:
        pickle.dump(pipe, f)
    print("Model saved as 'pipe.pkl'")
else:
    print("No model to save.")